# Setting up or sample 
We'll first take the first sample, from the first class type 0. 

In [ ]:
import numpy as np
import matplotlib as plt
data = np.load("../CodeTest/fashion_mnist_train.npz")
X, y = data["X"], data["y"]

data = np.load("../CodeTest/fashion_mnist_test.npz")
X_test, y_test = data["X"], data["y"]

np.set_printoptions(linewidth=120, threshold=np.inf, suppress=True)
sample = X[0]
subset = sample[:28, :28]
print(subset)

[[  0   0   0   0   0   1   0   0   0   0  41 188 103  54  48  43  87 168 133  16   0   0   0   0   0   0   0   0]
 [  0   0   0   1   0   0   0  49 136 219 216 228 236 255 255 255 255 217 215 254 231 160  45   0   0   0   0   0]
 [  0   0   0   0   0  14 176 222 224 212 203 198 196 200 215 204 202 201 201 201 209 218 224 164   0   0   0   0]
 [  0   0   0   0   0 188 219 200 198 202 198 199 199 201 196 198 198 200 200 200 200 201 200 225  41   0   0   0]
 [  0   0   0   0  51 219 199 203 203 212 238 248 250 245 249 246 247 252 248 235 207 203 203 222 140   0   0   0]
 [  0   0   0   0 116 226 206 204 207 204 101  75  47  73  48  50  45  51  63 113 222 202 206 220 224   0   0   0]
 [  0   0   0   0 200 222 209 203 215 200   0  70  98   0 103  59  68  71  49   0 219 206 214 210 250  38   0   0]
 [  0   0   0   0 247 218 212 210 215 214   0 254 243 139 255 174 251 255 205   0 215 217 214 208 220  95   0   0]
 [  0   0   0  45 226 214 214 215 224 205   0  42  35  60  16  17  12  13  70   

First, we must understand what we need to pass in whenever we have a convolutional block. In general the block consists of a the input image, a filter being slid onto the image where we iterate stride by stride, and an output feature map that is the result of the convolution operation at each given stride. Then, we apply an activation function, and finally we apply pooling to reduce the spatial dimensions for further layers. 
For now, we'll cover the convolutional block, and the forward pass before trying to tackle the backward pass of the model. 
Since for *my* project, I would like to use this model with color, input_shape is used for rgb values.

In [ ]:
class Conv_Layer:
    def __init__(self, input_shape, num_filters = 1, filter_size = (3, 3), strides = (1, 1), padding = "same"):

        self.input_shape = input_shape
        self.num_filters = num_filters
        self.filter_size = filter_size
        self.strides = strides
        self.padding = padding 
        self.biases = np.zeros((num_filters, 1)) 

        #We'll handle two scenarios, the first, where we pass in a (n, n, 1) or grayscale image, and a second
        #where we'll handle a (n, n, 3) or RGB image. 
        input_depth = input_shape[-1]
        n = self.filter_size[0] * self.filter_size[1] * input_depth
        std = np.sqrt(2.0 / n)
        
        #We can now do He initaliztion, we'll sample values from a standard distribution N (0, 1) and multiply it by our
        #std value to get N(0, std) 

        self.filter_weights = np.random.randn(
            filter_size[0],         #width
            filter_size[1],         #height
            input_depth,            #depth 
            num_filters             #number of filters
        )* std
    

Inside of our __init__ method, we have passed in a few things where:
* self: keyword to allow the object to save local copys of an data using the "self." prefix.
* input_shape: A list of 3 elements, (width, height, depth). Where width and height are the width and height of the input image, and depth being either 1 or 3, depending on if we're using greyscale or RGB, respectively. 
* num_filters: The number of unique filters we plan on passing through our model. In general, each unique filter should ideally learn to detect a different feature type. 
  * One filter might detect vertical edges.
  * One filter might detect horizontal edges.
  * One filter might detect diagonal edges.
  * Another filter may pick up on textures, corners, or blobs. 
* filter_size: A list for how large our *window* will be each time we slide it through our input image. Wtih a large filter size e.g (11, 11) we'd consider using *Fast Fourier Transform Convolution* to speedup our training time, since the overhead of using said algorithm is countered when our filter size is large. However, for smaller ones, I'll use pointwise multiplication (standard convolution operation). 
* strides: A list for how much the filter should move across each iteration horizontally and vertically. 
* padding: A string used to define how we'd like to set the padding of the image. Of course, using "same" padding will net us a border of a set size around the image such that we don't lose any edge detail. 

Next, we need to set the bias for each filter by doing:
```python
self.biases = np.zeros((num_filters, 1))
```
where we create a 2d array of zeros where each row corresponds to a unique filter, and since there is only ever one bias we denote that by using 1 as our column. 

Now, we need to handle the logic of initalizing the values inside of each filter. Since I used **He Inialization** for my dense layers, I will use the same initialziation here. We'll use the formula 
$$
\large W \rightarrow \mathcal{N} (0, \sqrt{\frac{2}{n}})
$$
where:
* $\mathcal{N} \rightarrow $ is the Gaussian probability distribution. 
* $0 \rightarrow $ is the mean of the distribution
* $\sqrt{\frac{2}{n}} \rightarrow$ is the standard deviation
* $n \rightarrow$ is the number of inputs to the node. 

You could think of n as the equation:
$$n = F \times F \times D 
$$
where:
* $F \rightarrow $ filter height and depth
* $D \rightarrow $ the number of input channels. 

```python
n = self.filter_size[0] * self.filter_size[1] * input_depth
std = np.sqrt(2.0 / n)
self.filter_weights = np.random.randn(filter_size[0],
                                       filter_size[1], 
                                              input_depth, num_filters) * std


# Forward Pass
Inside our forward pass we'll need to do the following things: 
1. Extract the input dimensions ($W_{in}, H_{in}, D_{in}$)
2. Calculate the output dimensions using the stride, padding, and fiter size: 
$$
W_{out} = \frac{W_{in} + 2P-F_W}{S_W} + 1
$$
$$
H_{out} = \frac{H_{in} + 2P-F_H}{S_H} + 1
$$
$$
D_{out} = numfilters
$$
 Notice how the output depth (number of channels in the output feature map), is equal to the number of filters the provided.
 Why?
 Each filter produces one channel in the output:
 Ex:
  * Input: $ 32 \times 32 \times 3 \text{RGB} $
    * So H_in = 32, W_in = 32, and D_in = 3
  * Filters: $ 8 \text{filters, each } 3 \times 3\ times 3 $
  * Output: $ 30 \times 30 \times 8 $ (assuming valid padding and stride = 1)



3. Pad input image (our borders of zeros per say)
4. Create an output tensor (our output feature map) with zeros using shape($H_{out}, W_{out}, D_{out}$)
5. Iterate over the spatial positions using the stride
6. In each position:
 * Slice the input window (create a small submatrix).
 * Perform elementwise multiplication between the filter and window.
 * sum the result plus bias
 * return this scaler to the output tensor.
7. Save the input image for backpropogation

In [1]:
class Conv_Layer:
    def __init__(self, input_shape, num_filters = 1, filter_size = (3, 3), strides = (1, 1), padding = "same"):

        self.input_shape = input_shape
        self.num_filters = num_filters
        self.filter_size = filter_size
        self.strides = strides
        self.padding = padding 
        self.biases = np.zeros((num_filters, 1)) 

        #We'll handle two scenarios, the first, where we pass in a (n, n, 1) or grayscale image, and a second
        #where we'll handle a (n, n, 3) or RGB image. 
        input_depth = input_shape[-1]
        n = self.filter_size[0] * self.filter_size[1] * input_depth
        std = np.sqrt(2.0 / n)
        
        #We can now do He initaliztion, we'll sample values from a standard distribution N (0, 1) and multiply it by our
        #std value to get N(0, std) 

        self.filter_weights = np.random.randn(
            filter_size[0],         #width
            filter_size[1],         #height
            input_depth,            #depth 
            num_filters             #number of filters
        )* std

    def forward(self, inputs):
        #Extract Input dimensions


    
        H_in, W_in, D_in = inputs.shape
        
        #Creating padding depending on padding = same, or padding = valid
        if self.padding == "same":
            P = (self.filter_size[0] - 1) // 2
        else:            
            P = 0

        #We need integer output dimensions, so cast equations to int
        H_out = int((H_in + 2 * P - self.filter_size[0]) / self.strides[0] + 1)
        W_out = int((W_in + 2 * P - self.filter_size[1]) / self.strides[1] + 1)
        
        #(P, P) -> pad top and bottom pixels by P pixels (axis 0)
        #(P, P) -> pad left and right pixels by P pixels (axis 1)
        #(0, 0) -> don't pad depth. 
        #contstant -> add constant_values for the padded values
        padded_inputs = np.pad(array = inputs, 
                            pad_width = ((P, P), (P, P), (0, 0)),
                            mode = 'constant',
                            constant_values = 0)
        
        

Conv_Layer.forward = forward

NameError: name 'forward' is not defined